# Setup and imports

In [ ]:
from google.colab import userdata     # Imports secret tokens for logins
from huggingface_hub import login     # Huggingface for fast implementation of transformers
import os
import sys
from pathlib import Path

''' Change snippet below with your file paths '''

file_path = ('/content/drive/My Drive/Thesis/belief-repr-1/')
sys.path.append(file_path)
if not os.path.exists('/content/drive/My Drive'):
    from google.colab import drive
    drive.mount('/content/drive')

''' Change snippet above with your file paths '''

secrets = {
    'hugging': userdata.get('hugging_token'),
    'wandb': userdata.get('wandb_api'),
    'nnsight': userdata.get('nnsight'),
    'openai': userdata.get('openai_api')
}

HF_TOKEN = secrets['hugging']
login(token=HF_TOKEN)

try:
    import transformer_lens as tlens
    import transformers as trans
    import openai
except:
    #  git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python
    # %pip install wandb -qU
    # %pip install -U bitsandbytes
    # %pip install -U accelerate
    %pip install -U transformers transformer_lens datasets einops jaxtyping
    %pip install openai
    import openai
    import transformer_lens as tlens
    import transformers as trans

"""
try:
    import sae_lens as slens
    from sae_lens import (
    SAE,
    ActivationsStore,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    SAEConfig,
    SAETrainingRunner,
    upload_saes_to_huggingface,
    )
    from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
    from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
except:
    # %pip install sae-lens
    #  git+https://github.com/callummcdougall/sae_vis.git@callum/v3
    %pip install openai>=1.56.2 nnsight
    %pip install --upgrade pydantic
    %pip install nnsight
"""

''' Navigate drive '''

import json
import pickle

''' For importing datasets '''

from datasets import load_dataset, Dataset

In [ ]:
# import nnsight

# import circuitsvis as cv

''' Tensor manipulation '''

import einops
from einops import einsum
import numpy as np
import torch as t                     # https://pytorch.org/docs/stable/torch.html
import torch.nn as nn                 # https://pytorch.org/docs/stable/nn.html
import torch.nn.functional as F       # https://pytorch.org/docs/stable/nn.functional.html

''' Utils '''

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from rich import print as rprint
from rich.table import Column, Table
from functools import partial         # We need this for fwd_hooks
import copy
from torch.amp import autocast
import gc
import seaborn as sns
import random

''' Should we need strong typing '''

from jaxtyping import Float, Int
from torch import Tensor
from typing import Callable, List, Tuple

''' For visualization of progress (in notebook) and training behavior (in wandb) '''

from tqdm import tqdm
import wandb                          # REMEMBER to log training loops if you want to analyze that behavior | how to: wandb.init() before training, wandb.log() at each epoch, wandb.finish() to clean cache
wandb.login(key=secrets['wandb'])
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

''' My things '''

from probes import SupervisedProbe, UnsupervisedProbe, probe_sweep
from visualization import *
from utils import *
from data_sets import *
from intervention import *

# Model and Config

In [ ]:
MODEL = 'llama'

def get_llama_legacy():

    tokenizer = trans.LlamaTokenizer.from_pretrained('huggyllama/llama-7b')
    hf_model = trans.LlamaForCausalLM.from_pretrained('huggyllama/llama-7b',
                                                      torch_dtype=t.float16,
                                                      device_map='cpu')
    model = tlens.HookedTransformer.from_pretrained(
        "llama-7b",
        hf_model=hf_model,
        device=t.device('cpu'),
        fold_ln=False,
        center_writing_weights=False,
        center_unembed=False,
        tokenizer=tokenizer
    ).half()

    return model
''' Dictionary for all of the models '''

mymodels = {
    'gemma': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b", device=t.device('cpu')).half(),
    'gemma_instruct': lambda: tlens.HookedTransformer.from_pretrained("gemma-2-9b-it", device=t.device('cpu')).half(),
    'gpt': lambda: tlens.HookedTransformer.from_pretrained("gpt2-large", device=device),
    'gpt-j': lambda: tlens.HookedTransformer.from_pretrained("EleutherAI/gpt-j-6B", device=t.device('cpu')).half(),
    'llama_legacy': get_llama_legacy,
    'llama': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B", device=t.device('cpu')).half(),
    'llama_instruct': lambda: tlens.HookedTransformer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device=t.device('cpu')).half(),
    'llama_medium': lambda: tlens.HookedTransformer.from_pretrained("Llama-2-13b", device=t.device('cpu')).half(),
    'llama_medium_instruct': lambda: tlens.HookedTransformer.from_pretrained("Llama-2-13b-chat", device=t.device('cpu')).half(),
    'pythia': lambda: tlens.HookedTransformer.from_pretrained("pythia-6.9b-deduped", device=t.device('cpu')).half(),
    'pythia_instruct': lambda: tlens.HookedTransformer.from_pretrained("pythia-6.9b", device=t.device('cpu')).half(),
    'yi': lambda: tlens.HookedTransformer.from_pretrained("yi-6b", device=t.device('cpu')).half(),
    'yi_instruct': lambda: tlens.HookedTransformer.from_pretrained("yi-6b-chat", device=t.device('cpu')).half()
}

model = mymodels[MODEL]()
model.to(device)

# TO DO: add SAE Hooked models

In [ ]:
# linear_layer best lr for residual stream on llama == ~1e-2/1e-3
# mlp best lr for residual stream on llama == ~1e-4
# mmp returns immediate results
# no batching is the fastest and works very well for cities.csv
# diminishing return after the 100th epoch for good activations

class ProbeConfig:
    def __init__(self):
        """ General """
        self.device = device
        self.batch_size_extractor = 32
        self.seed = 42
        """ Probe setup """
        self.probe_type = "linear" # Options: linear_layer | linear | mlp | mmp
        self.supervision = "S"
        self.direction_type = None # Options: linear | logistic | mmp | None
        self.verbose = True
        self.with_std = False
        self.var_normalize = True
        self.control = False
        """ Specs """
        self.batch_size = -1
        self.nepochs = 150
        self.ntries = 1
        self.lr = 1e-3
        self.weight_decay = 0.0
        self.dropout = 0.0
        self.C = 1e6
        self.max_iter = 500
        self.test_size = 0.1

probe_config = ProbeConfig()

# Experiments

## Patching

In [ ]:
model.to(t.device('cpu'))

In [ ]:
def compute_true_false_ratio(logits, model):
    """
    Compute the ratio P(True) / P(False) from logits, using multiple token variants.
    """
    # Variants for "True" and "False"
    true_tokens = ['true', 'True', 'TRUE', 'Ġtrue', 'ĠTrue', 'ĠTRUE']
    false_tokens = ['false', 'False', 'FALSE', 'Ġfalse', 'ĠFalse', 'ĠFALSE']

    # Convert to token IDs
    true_token_ids = [model.tokenizer.convert_tokens_to_ids(token) for token in true_tokens]
    false_token_ids = [model.tokenizer.convert_tokens_to_ids(token) for token in false_tokens]
    true_token_ids = [tid for tid in true_token_ids if tid != -1]
    false_token_ids = [fid for fid in false_token_ids if fid != -1]

    log_probs = t.nn.functional.log_softmax(logits, dim=-1)[0, -1]
    p_true = t.exp(log_probs[true_token_ids]).sum()
    p_false = t.exp(log_probs[false_token_ids]).sum()

    return t.log(p_true / p_false)


with t.autocast(device_type='cpu', dtype=t.float16):
  context = 'The city of Tokyo is in Japan. This statement is: True. \n The city of Hanoi is in Poland. This statement is: False. \n'
  clean_tokens = context + 'The city of Toronto is in Canada. This statement is: '
  corrupted_tokens = context + 'The city of Chicago is in Canada. This statement is: '
  corrupted_tokens = model.to_tokens(corrupted_tokens)
  model.add_hook("hook_embed", lambda tensor, hook: tensor.half())
  _, clean_cache = model.run_with_cache(clean_tokens)
  patching_result = tlens.patching.get_act_patch_resid_pre(model=model, corrupted_tokens=corrupted_tokens,
                                      clean_cache=clean_cache, patching_metric=compute_true_false_ratio,
                                      )

In [ ]:
tokz = ['The', 'City', 'of', 'Chicago/Toronto', 'is', 'in', 'Canada', '.', 'This', 'sentence', 'is', ':']

plt.figure(figsize=(12,2.5))
plt.imshow(cut.cpu(), cmap='Blues', interpolation='nearest', aspect='auto')
plt.colorbar(label='log P(True)/P(False)')
plt.title('Patching on post-MLP residual')
plt.xticks(ticks=range(len(tokz)), labels=tokz, rotation=45)
plt.xlabel('Column Index')
plt.ylabel('Layer')
plt.show()

## Get Activations

### Supervised

In [ ]:
# databuilder = ITIBuilder('tqa_gen')
# df = databuilder.get_dataset()
# df = pd.read_parquet("hf://datasets/truthfulqa/truthful_qa/generation/validation-00000-of-00001.parquet")
# df_split = df.iloc[:len(df) // 2]
# x, y, _ = get_prompts_tqa_gen(df_split)
# x = x[:-(len(x) % probe_config.batch_size_extractor)]
# y = y[:-(len(y) % probe_config.batch_size_extractor)]

databuilder = TrueFalseEasyBuilder()
dfs, df_all = databuilder.get_dataset()

# Preprocessing

df_all = df_all[df_all['filename'] != 'likely.csv']

# Trim the counterfact_true_false dataset to have balanced classes and a max of 1000 samples per class

target_df = df_all[df_all['filename'] == 'counterfact_true_false.csv']
label_0 = target_df[target_df['label'] == 0]
label_1 = target_df[target_df['label'] == 1]
retain_per_label = 1000
sampled_0 = label_0.sample(n=retain_per_label, random_state=42)
sampled_1 = label_1.sample(n=retain_per_label, random_state=42)
sampled_target_df = pd.concat([sampled_0, sampled_1])
df_all = pd.concat([df_all[df_all['filename'] != 'counterfact_true_false.csv'], sampled_target_df])
df_all['statement'] = df_all['statement'] + ' This sentence is: '

df_train, df_test = train_test_split(
    df_all,
    test_size=0.5,
    stratify=df_all['filename']
)
df_trimmed = df_train.iloc[:-(len(df_train) % probe_config.batch_size_extractor), :]
x = list(df_trimmed['statement'])
y = list(df_trimmed['label'])

# Extract & Probe

In [ ]:
"""
heads will be a list (layers) of lists (heads) of tensors (head) with shape n_batches d_batch d_head
attn_labels is a tensor of with shape n_batches d_batch
resid_activations will be a dict (activations) of tensors (activation) with shape n_batches d_batch d_model
resid_labels is a tensor of with shape n_batches d_batch
"""

# Heads
attn_activations, attn_labels = extract(model, x, y, device=device, batch_size=probe_config.batch_size_extractor, attn=True)
heads = [decompose_mha(x) for x in attn_activations.values()]

# Residual stream
resid_activations, resid_labels = extract(model, x, y, device=device, batch_size=probe_config.batch_size_extractor, attn=False)

model.to(t.device('cpu'))
gc.collect()
t.cuda.empty_cache()

## Playing with KDE

In [ ]:
top_heads, top_values = get_top_heads(attn_activations, 5)

In [ ]:
from sklearn.linear_model import LogisticRegression

data = heads[15][2]
labels = 1 - attn_labels

probe = LogisticRegression(max_iter=probe_config.max_iter,
                                            solver="lbfgs",
                                            C=probe_config.C,
                                            random_state=probe_config.seed,
                                            n_jobs=-1)

kde(data=data, labels=labels, model=probe,
    n_dir=2, zoom_strength=0.5, adjust=0.01,
    kernel=False, scatter=True, pca=False)

In [ ]:
# Plot PCA'd data

kde(data=data, labels=labels, model=probe,
    n_dir=2, zoom_strength=0.5, adjust=0.01,
    kernel=False, scatter=True, pca=True)

## Probe Sweeps

In [ ]:
probe_config.var_normalize = False
probe_config.verbose = False
tot_accuracies_heads = []
tot_directions_heads = []
tot_probes_heads = []

for layer in tqdm(range(len(heads)), desc="Layers"):
  accuracies, directions, probes = probe_sweep(heads[layer], attn_labels, probe_config)
  tot_accuracies_heads.append(accuracies)
  tot_directions_heads.append(directions)
  tot_probes_heads.append(probes)

tot_accuracies_heads = np.array(tot_accuracies_heads)

top_heads, top_values = get_top_heads(tot_accuracies_heads, 5)
print(top_heads, top_values)

In [ ]:
pretty_heatmap(tot_accuracies_heads,
               title=f"Probe Accuracy per Head per Layer (Sorted). Best acc: {top_values[0]:.3f}",
               x_axis="Heads (Sorted by Probe Accuracy)",
               y_axis="Layers (Bottom-Up)",
               model=f'{MODEL}',
               probe=probe_config.probe_type,
               dataset="TrueFalse")

In [ ]:
# For instance

probe_config.lr = 1e-4 * 5
probe_config.dropout = 0.1
probe_config.batch_size = 256
probe_config.control = False
probe_config.probe_type = 'mlp'

accuracies, directions, _ = probe_sweep(resid_activations.values(), resid_labels, probe_config)

pretty_line(x=accuracies,
            title=f"Model: {MODEL} | Probe: {probe_config.probe_type}, best accuracy: {np.array(accuracies).max():.3f} | Dataset: TrueFalse",
            x_axis="Layers",
            y_axis="Accuracy",
            x_label="Residual stream, mid",
            y_label="Accuracy")

# Intervene

In [ ]:
# attn

with open(f'{file_path}{MODEL}_direction_{probe_config.direction_type}.pkl', 'rb') as file:
    directions = pickle.load(file)
with open(f'{file_path}{MODEL}_accuracies_{probe_config.direction_type}.pkl', 'rb') as file:
    accuracies = pickle.load(file)

directions = t.stack([t.stack(sublist, dim=0) for sublist in directions], dim=0)

In [ ]:
# resid

with open(f'{file_path}{MODEL}_line_accuracies_{probe_config.direction_type}.pkl', 'rb') as file:
    accuracies = pickle.load(file)
with open(f'{file_path}{MODEL}_line_directions_{probe_config.direction_type}.pkl', 'rb') as file:
    directions = pickle.load(file)

random_directions = [t.rand_like(x, dtype=t.float) * 0.2 - 0.1 for x in directions]

In [ ]:
df_true = df_test[df_test['label'] == 1]
df_false = df_test[df_test['label'] == 0]

In [ ]:
test_sentences = list(df_true['statement'])
test_sample = random.sample(
                            test_sentences,
                            # len(test_sentences)               # Actual test
                            100                                 # For sweeping purposes 
                            )
sampled_df = df_true[df_true['statement'].isin(test_sample)]
prompts = list(sampled_df['statement'])
labels = list(sampled_df['label'])

In [ ]:
model.to(device)
model.eval()

''' == PARAMETER SWEEP == '''

best_k = 76
best_a = 10

strength = np.abs(best_k*best_a/(model.cfg.n_heads*model.cfg.n_layers)) # Attn
# strength = np.abs(best_k*best_a/model.cfg.n_layers) # Resid
print("Intervention Strength: ", strength)

with autocast('cuda'):
  metric = 'boolprobs'
  
  ''' Find params through sweeping '''
  ks = [1, 2, 3, 4, 5]
  alphas = [0, -2, -3, -4, -5] if labels[0] == 1 else [0, 2, 3, 4, 5]     # We steer towards false if we have the true dataset and vice versa
  ''' Once you have found the best params, use them below with the full dataset'''
  # ks = [best_k]
  # alphas = [0, best_a]                                                  # We keep alpha=0 to see the original accuracy

  for_sweep = parameter_sweep(
      model,
      prompts,
      accuracies,
      random_directions,
      ks=ks,
      alphas=alphas,
      metric=metric,
      secret=secrets['openai'],
      labels=labels,
      attn=False
  )

In [ ]:
boolp, probdiff = for_sweep

pretty_sweep(
             probdiff,              # or boolp
             ks=ks,
             alphas=alphas,
             metric=metric
             )